# IS-218 Oppgave 2: Befolkning vs. tilfluktsrom

Dette notatet laster geodata for befolkning og tilfluktsrom i Norge, analyserer kapasitet innen en valgt radius, og viser resultatet interaktivt i kart.

## 1) Installer og importer biblioteker

Denne cellen installerer eventuelle manglende pakker og importerer bibliotekene vi trenger for nedlasting, geodata-analyse, interaktivt kart og eksport.

In [1]:
import importlib
import subprocess
import sys

required_packages = [
    "pandas",
    "geopandas",
    "folium",
    "ipywidgets",
    "openpyxl",
    "requests",
]

for package in required_packages:
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import json
import zipfile
from pathlib import Path
from datetime import datetime

import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import LocateControl
from branca.element import MacroElement, Template
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests

In [2]:
# Datakilder fra oppgaveteksten
POP_GPKG_URL = "https://kart.ssb.no/api/filestorage/v1/files/069b07bf-c1cc-79bd-8000-60a4569dc422/2026-03-10-befolkning_250m_2025.gpkg?accesskey=fsXGJFHwmLDJBLcKiZERYGYccvFybhy3"
POP_GEOJSON_URL = "https://kart.ssb.no/api/filestorage/v1/files/069b07eb-9b71-795a-8000-c57506f83ff9/2026-03-10-befolkning_250m_2025.geojson?accesskey=ZWmKOwVHQdcaeeKncdwBjpQhRrv2tx76"
SHELTER_ZIP_URL = "https://nedlasting.geonorge.no/api/download/order/4d44140d-5ff1-4dfd-ad0a-51bf93286076/d3516669-a1cb-46ff-aaad-90cfaab19346"
SHELTER_GML_ZIP_URL = "https://nedlasting.geonorge.no/api/download/order/4d44140d-5ff1-4dfd-ad0a-51bf93286076/539e16d2-391c-4bb7-9a25-18a199a4a4ab"
POP_GML_ZIP_URL = "https://nedlasting.geonorge.no/api/download/order/4d44140d-5ff1-4dfd-ad0a-51bf93286076/3300bbc2-94a9-4773-98d3-f86878f00fe7"
SHELTER_WMS_URL = "https://ogc.dsb.no/wms.ashx"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def download_file(url: str, out_path: Path) -> Path:
    """Laster ned fil hvis den ikke finnes fra før."""
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path

    response = requests.get(url, timeout=120)
    response.raise_for_status()
    out_path.write_bytes(response.content)
    return out_path


def read_any_vector(path: Path, preferred_exts=(".geojson", ".gpkg", ".gml", ".json")) -> gpd.GeoDataFrame:
    """Leser første støttede geodatafil i en mappe/zip."""
    if path.is_file() and path.suffix.lower() in preferred_exts:
        return gpd.read_file(path)

    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path, "r") as zf:
            extract_dir = DATA_DIR / path.stem
            extract_dir.mkdir(exist_ok=True)
            zf.extractall(extract_dir)
        path = extract_dir

    candidates = []
    for ext in preferred_exts:
        candidates.extend(path.rglob(f"*{ext}"))

    if not candidates:
        raise FileNotFoundError(f"Fant ingen geodatafiler i {path}")

    return gpd.read_file(candidates[0])

## 2) Last ned og klargjor data

Denne delen laster ned befolknings- og tilfluktsromdata (med fallback-kilder), finner relevante kolonner for befolkning og kapasitet, og standardiserer datasett for analyse.

In [3]:
def find_numeric_column(df: gpd.GeoDataFrame, preferred_names):
    lower_map = {c.lower(): c for c in df.columns}

    for name in preferred_names:
        if name in lower_map:
            return lower_map[name]

    for c in df.columns:
        cl = c.lower()
        if any(token in cl for token in preferred_names):
            return c

    numeric_candidates = [
        c for c in df.columns
        if c != "geometry" and pd.api.types.is_numeric_dtype(df[c])
    ]
    if not numeric_candidates:
        raise ValueError("Fant ingen numeriske kandidatkolonner")
    return numeric_candidates[0]


# Last ned filer lokalt
pop_gpkg_path = download_file(POP_GPKG_URL, DATA_DIR / "befolkning_250m_2025.gpkg")
pop_geojson_path = download_file(POP_GEOJSON_URL, DATA_DIR / "befolkning_250m_2025.geojson")
shelter_zip_path = download_file(SHELTER_ZIP_URL, DATA_DIR / "tilfluktsrom_geojson.zip")
shelter_gml_zip_path = download_file(SHELTER_GML_ZIP_URL, DATA_DIR / "tilfluktsrom_gml.zip")

# Last befolkning (prioriter gpkg, fallback geojson)
try:
    population_gdf = gpd.read_file(pop_gpkg_path)
except Exception:
    population_gdf = gpd.read_file(pop_geojson_path)

# Last tilfluktsrom (prioriter geojson-zip, fallback gml-zip)
try:
    shelters_gdf = read_any_vector(shelter_zip_path, preferred_exts=(".geojson", ".json", ".gml"))
except Exception:
    shelters_gdf = read_any_vector(shelter_gml_zip_path, preferred_exts=(".gml", ".geojson", ".json"))

# Finn kolonne med befolkningstall
pop_col = find_numeric_column(
    population_gdf,
    preferred_names=["befolkning", "personer", "population", "pop", "antall"]
)

# Finn kolonne med kapasitet i tilfluktsrom
cap_col = find_numeric_column(
    shelters_gdf,
    preferred_names=["kapasitet", "capacity", "plasser", "antall", "personer"]
)

# Gi stabile feltnavn
population_gdf = population_gdf[[pop_col, "geometry"]].copy()
population_gdf.columns = ["population", "geometry"]
population_gdf["population"] = pd.to_numeric(population_gdf["population"], errors="coerce").fillna(0)

shelters_gdf = shelters_gdf[[cap_col, "geometry"]].copy()
shelters_gdf.columns = ["capacity", "geometry"]
shelters_gdf["capacity"] = pd.to_numeric(shelters_gdf["capacity"], errors="coerce").fillna(0)

# Fjern ugyldige geometrier og rader uten kapasitet/befolkning
population_gdf = population_gdf[population_gdf.geometry.notnull()].copy()
shelters_gdf = shelters_gdf[shelters_gdf.geometry.notnull()].copy()

population_gdf = population_gdf[population_gdf["population"] > 0].copy()
shelters_gdf = shelters_gdf[shelters_gdf["capacity"] > 0].copy()

# CRS-harmonisering
if population_gdf.crs is None:
    population_gdf = population_gdf.set_crs(4326)
if shelters_gdf.crs is None:
    shelters_gdf = shelters_gdf.set_crs(4326)

population_wgs84 = population_gdf.to_crs(4326)
shelters_wgs84 = shelters_gdf.to_crs(4326)

# ETRS89 / LAEA Europe for meter-baserte avstander
population_metric = population_wgs84.to_crs(3035)
shelters_metric = shelters_wgs84.to_crs(3035)

# Bruk punktrepresentasjon for populasjonsruter i radiusberegning
population_points_metric = population_metric.copy()
population_points_metric["geometry"] = population_points_metric.geometry.centroid

# Stabil ID for shelters
shelters_metric = shelters_metric.reset_index(drop=True)
shelters_metric["shelter_id"] = shelters_metric.index + 1
shelters_wgs84 = shelters_metric.to_crs(4326)

print(f"Population cells loaded: {len(population_points_metric):,}")
print(f"Shelters loaded: {len(shelters_metric):,}")
print(f"Population column used: {pop_col}")
print(f"Shelter capacity column used: {cap_col}")

Population cells loaded: 224,761
Shelters loaded: 555
Population column used: pop_tot
Shelter capacity column used: plasser


## 3) Beregn dekning innen radius

Denne cellen lager en funksjon som for hver bunker/tilfluktsrom summerer befolkningen innenfor valgt radius, vurderer om kapasiteten er nok, og returnerer en resultat-tabell.

In [4]:
def compute_coverage(radius_m: int) -> pd.DataFrame:
    """Summerer befolkning innenfor radius for hvert tilfluktsrom."""
    buffers = shelters_metric[["shelter_id", "capacity", "geometry"]].copy()
    buffers["geometry"] = buffers.geometry.buffer(radius_m)

    # Spatial join: hvilke befolkningspunkter faller innenfor hvert buffer
    joined = gpd.sjoin(
        population_points_metric[["population", "geometry"]],
        buffers,
        how="inner",
        predicate="within",
    )

    pop_by_shelter = (
        joined.groupby("shelter_id", as_index=False)["population"]
        .sum()
        .rename(columns={"population": "population_within_radius"})
    )

    result = shelters_metric[["shelter_id", "capacity", "geometry"]].merge(
        pop_by_shelter,
        on="shelter_id",
        how="left",
    )
    result["population_within_radius"] = result["population_within_radius"].fillna(0)
    result["enough_capacity"] = result["capacity"] >= result["population_within_radius"]
    result["missing_capacity"] = (
        result["population_within_radius"] - result["capacity"]
    ).clip(lower=0)

    result_wgs84 = gpd.GeoDataFrame(result, geometry="geometry", crs=shelters_metric.crs).to_crs(4326)
    result_wgs84["lat"] = result_wgs84.geometry.y
    result_wgs84["lon"] = result_wgs84.geometry.x

    ordered_cols = [
        "shelter_id",
        "capacity",
        "population_within_radius",
        "enough_capacity",
        "missing_capacity",
        "lat",
        "lon",
        "geometry",
    ]
    return result_wgs84[ordered_cols]

# Eksempelberegning for kontroll
preview_df = compute_coverage(radius_m=1000)
preview_df.head(5)

,shelter_id,capacity,population_within_radius,enough_capacity,missing_capacity,lat,lon,geometry
0,1,400,6347.0,False,5947.0,59.404915,10.462108,POINT (10.46211 59.40491)
1,2,390,6540.0,False,6150.0,59.417183,10.483879,POINT (10.48388 59.41718)
2,3,370,7518.0,False,7148.0,59.417700,10.479063,POINT (10.47906 59.4177)
3,4,215,6782.0,False,6567.0,59.415724,10.483809,POINT (10.48381 59.41572)
4,5,800,4173.0,False,3373.0,59.489427,10.315461,POINT (10.31546 59.48943)


## 4) Interaktivt kart, lagvalg og eksport

Denne cellen bygger et interaktivt kart med radius-slider, valgbare datalag, fargekoding av tilfluktsrom (gronn = nok kapasitet, rod = ikke nok), tabellvisning og knapper for eksport til CSV/XLSX.

In [5]:
radius_slider = widgets.IntSlider(
    value=1000,
    min=250,
    max=5000,
    step=250,
    description="Radius (m)",
    continuous_update=False,
)

show_population_chk = widgets.Checkbox(value=True, description="Vis befolkningsruter")
show_buffers_chk = widgets.Checkbox(value=False, description="Vis radius-buffere")
show_wms_chk = widgets.Checkbox(value=False, description="Vis DSB WMS-lag")

export_csv_btn = widgets.Button(description="Eksporter CSV", button_style="success")
export_xlsx_btn = widgets.Button(description="Eksporter XLSX", button_style="info")

controls = widgets.HBox([
    radius_slider,
    show_population_chk,
    show_buffers_chk,
    show_wms_chk,
])
exports = widgets.HBox([export_csv_btn, export_xlsx_btn])

map_output = widgets.Output()
table_output = widgets.Output()
status_output = widgets.Output()
summary_output = widgets.Output()

state = {"latest_table": pd.DataFrame()}


def add_user_location_recommendation(m: folium.Map, result_df: gpd.GeoDataFrame):
    """Legger til logikk i kartet for a finne naermeste og enkleste tilfluktsrom fra brukerens posisjon."""
    shelters_payload = result_df[
        [
            "shelter_id",
            "capacity",
            "population_within_radius",
            "enough_capacity",
            "missing_capacity",
            "lat",
            "lon",
        ]
    ].copy()

    shelters_json = shelters_payload.to_json(orient="records")
    map_name = m.get_name()

    template = f"""
    {{% macro script(this, kwargs) %}}
    var map = {map_name};
    var shelters = {shelters_json};
    var recommendationLayer = L.layerGroup().addTo(map);

    function haversineMeters(lat1, lon1, lat2, lon2) {{
        var R = 6371000;
        var dLat = (lat2 - lat1) * Math.PI / 180;
        var dLon = (lon2 - lon1) * Math.PI / 180;
        var a = Math.sin(dLat / 2) * Math.sin(dLat / 2) +
                Math.cos(lat1 * Math.PI / 180) * Math.cos(lat2 * Math.PI / 180) *
                Math.sin(dLon / 2) * Math.sin(dLon / 2);
        var c = 2 * Math.atan2(Math.sqrt(a), Math.sqrt(1 - a));
        return R * c;
    }}

    function rankShelters(userLat, userLon) {{
        if (!shelters.length) {{
            return null;
        }}

        shelters.forEach(function(s) {{
            s.distance_m = haversineMeters(userLat, userLon, s.lat, s.lon);
        }});

        shelters.sort(function(a, b) {{ return a.distance_m - b.distance_m; }});
        var nearest = shelters[0];
        var enough = shelters.filter(function(s) {{ return s.enough_capacity; }});
        var easiest = enough.length ? enough[0] : nearest;
        return {{ nearest: nearest, easiest: easiest }};
    }}

    function renderRecommendation(userLat, userLon) {{
        recommendationLayer.clearLayers();

        var ranked = rankShelters(userLat, userLon);
        if (!ranked) {{
            return;
        }}

        var nearest = ranked.nearest;
        var easiest = ranked.easiest;

        var userMarker = L.circleMarker([userLat, userLon], {{
            radius: 7,
            color: '#1f77b4',
            fillColor: '#1f77b4',
            fillOpacity: 1
        }}).bindPopup('Din posisjon').addTo(recommendationLayer);

        var nearestMarker = L.circleMarker([nearest.lat, nearest.lon], {{
            radius: 7,
            color: '#6a3d9a',
            fillColor: '#6a3d9a',
            fillOpacity: 1
        }}).bindPopup(
            'Naermeste tilfluktsrom<br>' +
            'ID: ' + nearest.shelter_id + '<br>' +
            'Avstand: ' + Math.round(nearest.distance_m) + ' m'
        ).addTo(recommendationLayer);

        L.polyline([[userLat, userLon], [nearest.lat, nearest.lon]], {{
            color: '#6a3d9a',
            weight: 2,
            dashArray: '5, 6'
        }}).addTo(recommendationLayer);

        var easiestColor = easiest.enough_capacity ? '#2ca25f' : '#ff7f0e';
        var easiestTitle = easiest.enough_capacity ? 'Enkleste med nok kapasitet' : 'Beste tilgjengelige (ingen med nok kapasitet)';

        var easiestMarker = L.circleMarker([easiest.lat, easiest.lon], {{
            radius: 8,
            color: easiestColor,
            fillColor: easiestColor,
            fillOpacity: 1
        }}).bindPopup(
            easiestTitle + '<br>' +
            'ID: ' + easiest.shelter_id + '<br>' +
            'Avstand: ' + Math.round(easiest.distance_m) + ' m<br>' +
            'Kapasitet: ' + Math.round(easiest.capacity) + '<br>' +
            'Befolkning i radius: ' + Math.round(easiest.population_within_radius) + '<br>' +
            'Manglende kapasitet: ' + Math.round(easiest.missing_capacity)
        ).addTo(recommendationLayer);

        if (nearest.shelter_id !== easiest.shelter_id) {{
            L.polyline([[userLat, userLon], [easiest.lat, easiest.lon]], {{
                color: easiestColor,
                weight: 3
            }}).addTo(recommendationLayer);
        }}

        userMarker.openPopup();
    }}

    map.on('locationfound', function(e) {{
        renderRecommendation(e.latlng.lat, e.latlng.lng);
    }});

    map.on('locationerror', function() {{
        alert('Kunne ikke hente posisjon. Sjekk nettleserinnstillinger for lokasjon.');
    }});
    {{% endmacro %}}
    """

    macro = MacroElement()
    macro._template = Template(template)
    m.get_root().add_child(macro)


def make_map(result_df: gpd.GeoDataFrame, radius_m: int, show_population: bool, show_buffers: bool, show_wms: bool):
    center = [64.5, 15.0]
    m = folium.Map(location=center, zoom_start=5, tiles="CartoDB positron")

    if show_wms:
        folium.WmsTileLayer(
            url=SHELTER_WMS_URL,
            layers="0",
            name="DSB Tilfluktsrom WMS",
            fmt="image/png",
            transparent=True,
            overlay=True,
            control=True,
            attr="DSB",
        ).add_to(m)

    if show_population:
        pop_preview = population_wgs84.copy()
        if len(pop_preview) > 40000:
            pop_preview = pop_preview.sample(40000, random_state=42)

        folium.GeoJson(
            pop_preview.to_json(),
            name="Befolkning (ruter)",
            style_function=lambda x: {
                "fillColor": "#2b8cbe",
                "color": "#2b8cbe",
                "weight": 0.2,
                "fillOpacity": 0.1,
            },
        ).add_to(m)

    if show_buffers:
        result_metric = result_df.to_crs(3035)
        buffer_layer = result_metric.copy()
        buffer_layer["geometry"] = buffer_layer.geometry.buffer(radius_m)
        buffer_wgs = buffer_layer.to_crs(4326)

        folium.GeoJson(
            buffer_wgs.to_json(),
            name=f"Buffer ({radius_m} m)",
            style_function=lambda x: {
                "fillColor": "#636363",
                "color": "#636363",
                "weight": 0.5,
                "fillOpacity": 0.08,
            },
        ).add_to(m)

    shelter_layer = folium.FeatureGroup(name="Tilfluktsrom", show=True)

    for _, row in result_df.iterrows():
        color = "green" if row["enough_capacity"] else "red"
        popup_text = (
            f"Shelter ID: {int(row['shelter_id'])}<br>"
            f"Capacity: {int(row['capacity'])}<br>"
            f"Population in radius: {int(row['population_within_radius'])}<br>"
            f"Enough capacity: {'Ja' if row['enough_capacity'] else 'Nei'}<br>"
            f"Missing capacity: {int(row['missing_capacity'])}"
        )

        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=4,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.9,
            popup=folium.Popup(popup_text, max_width=350),
        ).add_to(shelter_layer)

    shelter_layer.add_to(m)

    LocateControl(
        auto_start=False,
        flyTo=True,
        keepCurrentZoomLevel=True,
        position="topleft",
        strings={"title": "Vis min posisjon og finn naermeste tilfluktsrom"},
    ).add_to(m)
    add_user_location_recommendation(m, result_df)

    folium.LayerControl(collapsed=False).add_to(m)
    return m


def refresh_view(*_):
    radius = radius_slider.value
    show_population = show_population_chk.value
    show_buffers = show_buffers_chk.value
    show_wms = show_wms_chk.value

    result_df = compute_coverage(radius)
    state["latest_table"] = pd.DataFrame(result_df.drop(columns="geometry"))

    # Calculate coverage statistics
    covered_pop = result_df[result_df["enough_capacity"]]["population_within_radius"].sum()
    uncovered_pop = result_df[~result_df["enough_capacity"]]["population_within_radius"].sum()
    total_covered = covered_pop + uncovered_pop
    total_population = population_wgs84["population"].sum()
    coverage_pct = (total_covered / total_population * 100) if total_population > 0 else 0
    inadequate_capacity = result_df["missing_capacity"].sum()

    # Display summary statistics
    with summary_output:
        clear_output(wait=True)
        summary_html = f"""
        <div style="background-color:#f0f0f0; padding:15px; border-radius:8px; border-left:4px solid #1f77b4;">
            <h3 style="margin-top:0; color:#1f77b4;">Dekkingsstatistikk (Radius: {radius} m)</h3>
            <table style="width:100%; border-collapse:collapse;">
                <tr style="background-color:#e8f4f8;">
                    <td style="padding:8px; border:1px solid #ddd;"><b>Befolkning med tilstrekkelig kapasitet:</b></td>
                    <td style="padding:8px; border:1px solid #ddd; text-align:right; color:#27ae60;"><b>{int(covered_pop):,}</b></td>
                </tr>
                <tr>
                    <td style="padding:8px; border:1px solid #ddd;"><b>Befolkning uten tilstrekkelig kapasitet:</b></td>
                    <td style="padding:8px; border:1px solid #ddd; text-align:right; color:#e74c3c;"><b>{int(uncovered_pop):,}</b></td>
                </tr>
                <tr style="background-color:#e8f4f8;">
                    <td style="padding:8px; border:1px solid #ddd;"><b>Totalt dekket innen radius:</b></td>
                    <td style="padding:8px; border:1px solid #ddd; text-align:right;"><b>{int(total_covered):,}</b></td>
                </tr>
                <tr>
                    <td style="padding:8px; border:1px solid #ddd;"><b>Samlet befolkning i Norge:</b></td>
                    <td style="padding:8px; border:1px solid #ddd; text-align:right;"><b>{int(total_population):,}</b></td>
                </tr>
                <tr style="background-color:#fff3cd;">
                    <td style="padding:8px; border:1px solid #ddd;"><b>Dekning % av total befolkning:</b></td>
                    <td style="padding:8px; border:1px solid #ddd; text-align:right;"><b>{coverage_pct:.1f}%</b></td>
                </tr>
                <tr>
                    <td style="padding:8px; border:1px solid #ddd;"><b>Manglende kapasitetsplass:</b></td>
                    <td style="padding:8px; border:1px solid #ddd; text-align:right; color:#e67e22;"><b>{int(inadequate_capacity):,}</b></td>
                </tr>
            </table>
        </div>
        """
        from IPython.display import HTML
        display(HTML(summary_html))

    with map_output:
        clear_output(wait=True)
        m = make_map(result_df, radius, show_population, show_buffers, show_wms)
        display(m)

    with table_output:
        clear_output(wait=True)
        summary = state["latest_table"].copy()
        summary["enough_capacity"] = summary["enough_capacity"].map({True: "Ja", False: "Nei"})
        display(summary)


def export_csv(_):
    if state["latest_table"].empty:
        return
    file_name = f"shelter_coverage_{radius_slider.value}m_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    state["latest_table"].to_csv(file_name, index=False)
    with status_output:
        clear_output(wait=True)
        print(f"CSV exportert: {file_name}")


def export_xlsx(_):
    if state["latest_table"].empty:
        return
    file_name = f"shelter_coverage_{radius_slider.value}m_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    state["latest_table"].to_excel(file_name, index=False)
    with status_output:
        clear_output(wait=True)
        print(f"XLSX exportert: {file_name}")


radius_slider.observe(refresh_view, names="value")
show_population_chk.observe(refresh_view, names="value")
show_buffers_chk.observe(refresh_view, names="value")
show_wms_chk.observe(refresh_view, names="value")
export_csv_btn.on_click(export_csv)
export_xlsx_btn.on_click(export_xlsx)

display(controls)
display(exports)
display(status_output)
display(summary_output)
display(map_output)
display(table_output)

refresh_view()

Output()

Output()

Output()

Output()